In [66]:
from dotenv import load_dotenv
load_dotenv()

True

In [67]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

from langchain_groq import ChatGroq

In [68]:
loader = PyMuPDFLoader(file_path="../data/Computer-Network-Notes.pdf")
docs = loader.load()

In [69]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 600, chunk_overlap = 200)
docs = splitter.split_documents(docs)

In [70]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLm-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3737.04it/s]


In [71]:
vector_store = InMemoryVectorStore.from_documents(
    documents=docs,
    embedding=embeddings

)

In [72]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [73]:
query = "What is the difference between TCP and IP?"
chunks = vector_store.similarity_search(query)
contexts =""
for chunk in chunks:
    contexts = chunk.page_content + "\n\n"

prompt = f"""
        You are a smart RAG system. You always provide the answer for users questions based on the provided context:

        Question:{query}

        Content: {contexts}

"""

res = llm.invoke(prompt)

print(res.content)

**TCP vs. IP – the main differences**

| Feature | **IP (Internet Protocol)** | **TCP (Transmission Control Protocol)** |
|---------|-----------------------------|----------------------------------------|
| **Layer** | Network layer (Layer 3 of the OSI model) | Transport layer (Layer 4) |
| **Purpose** | Provides logical addressing and routing of packets from source to destination across networks. | Provides reliable, ordered delivery of data between end‑to‑end hosts. |
| **Reliability** | No reliability guarantees – packets may be lost, duplicated or arrive out of order. | Reliable: guarantees delivery, ordering, retransmission of lost segments. |
| **Connection** | Connection‑less: each packet is independent. | Connection‑oriented: establishes a session (handshake) before data transfer. |
| **Addressing** | Uses a 32‑bit (IPv4) or 128‑bit (IPv6) address to identify hosts. | Uses 16‑bit port numbers (together with IP addresses) to identify individual applications/processes on a host. 

#### Multi Documents Rag

#### Pdf Docs already exists

In [74]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(web_path="https://qdrant.tech/documentation/overview/")
docs2 = loader.load()

In [75]:
docs2 = splitter.split_documents(docs2)

In [76]:
len(docs2)

43

In [77]:
multi_vector_store = InMemoryVectorStore.from_documents(documents=docs2, embedding=embeddings)

In [78]:
query = "What is Qdrant Retrieval Process?"
chunks = multi_vector_store.similarity_search(query)
contexts =""
for chunk in chunks:
    contexts = chunk.page_content + "\n\n"

prompt = f"""
        You are a smart RAG system. You always provide the answer for users questions based on the provided context:

        Question:{query}

        Content: {contexts}

"""

res = llm.invoke(prompt)

print(res.content)

**Qdrant Retrieval Process (based on the provided context)**  

- **In‑memory storage for speed** – By default, Qdrant keeps all vectors in RAM so that similarity queries can be answered with minimal latency.  
- **Caching of frequently accessed vectors** – Vectors that are queried often stay cached in memory, ensuring that repeated look‑ups are fast.  
- **Optional disk off‑loading** – When collections grow to millions of vectors, keeping everything in RAM can become costly. Qdrant allows you to offload data to disk on the fly, even for existing collections. This reduces memory usage while still keeping the most‑used vectors cached for quick retrieval.  

So, the retrieval process relies on an in‑memory vector store for rapid similarity search, with a caching layer for hot vectors and an optional disk‑backed fallback to manage larger datasets.
